通过调用 graph.get_state(config) 来查看图的最新状态。

In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    foo: str
    bar: Annotated[list[str], add]

def node_a(state: State):
    return {"foo": "a", "bar": ["a"]}

def node_b(state: State):
    return {"foo": "b", "bar": ["b"]}


workflow = StateGraph(State)
workflow.add_node(node_a)
workflow.add_node(node_b)
workflow.add_edge(START, "node_a")
workflow.add_edge("node_a", "node_b")
workflow.add_edge("node_b", END)

checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}
graph.invoke({"foo": "", "bar":[]}, config)

{'foo': 'b', 'bar': ['a', 'b']}

In [2]:
snapshot = graph.get_state(config)
snapshot

StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f14550c-ea34-6a56-8002-20ee429917ea'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-05-01T11:27:59.170407+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f14550c-ea33-6886-8001-8a6ea0cf9e52'}}, tasks=(), interrupts=())

可以通过调用 graph.get_state_history(config) 来获取给定线程的图执行完整历史记录。

In [3]:
history = list(graph.get_state_history(config))
history

[StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f14550c-ea34-6a56-8002-20ee429917ea'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-05-01T11:27:59.170407+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f14550c-ea33-6886-8001-8a6ea0cf9e52'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'foo': 'a', 'bar': ['a']}, next=('node_b',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f14550c-ea33-6886-8001-8a6ea0cf9e52'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-05-01T11:27:59.169949+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f14550c-ea31-6e00-8000-6522f005c0bd'}}, tasks=(PregelTask(id='17e02b43-eb90-1c29-f13b-8b7c255b9370', name='node_b', path=('__pregel_pull', 'node_b'), error=None, interrupts

可以筛选状态历史记录，以查找符合特定条件的检查点

In [5]:
# Find the checkpoint before a specific node executed
before_node_b = next(s for s in history if s.next == ("node_b",))

# Find a checkpoint by step number
step_2 = next(s for s in history if s.metadata["step"] == 2) # type: ignore

# Find checkpoints created by update_state
forks = [s for s in history if s.metadata["source"] == "update"] # type: ignore